# 🌾 Wheat Plant Disease: Model Development on Clean Dataset
**Source Pipeline:** [`ssahildhin/wheat-disease-detection`](https://www.kaggle.com/code/ssahildhin/wheat-disease-detection)
**Objective:** Build and train both a Baseline Custom CNN and a Pretrained Fine-Tuned CNN (MobileNetV2) on the clean, deduplicated dataset.

---
### 📋 Steps in this Notebook:
1. **Install dependencies first.**
2. **Load clean dataset splits** (`clean_train.csv`, `clean_val.csv`, `clean_test.csv`).
3. **Construct high-performance `tf.data.Dataset` pipelines** with parallel prefetching and data augmentation.
4. **Compute balanced class weights** to counter class imbalances.
5. **Build & Train Baseline Custom CNN** (3 Conv blocks + `GlobalAveragePooling2D`).
6. **Build & Fine-Tune Pretrained CNN (MobileNetV2)**:
   - Phase 1: Feature Extraction (frozen base, $lr=10^{-3}$)
   - Phase 2: Deep Fine-Tuning (top 30 layers unfrozen, $lr=10^{-5}$)
7. **Save models & histories** for performance evaluation.

In [ ]:
# Step 1: Install dependencies
!pip install -q tensorflow numpy pandas matplotlib seaborn scikit-learn pillow
print('✅ Dependencies installed successfully!')


In [ ]:
# Step 2: Import libraries
import os
import json
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

print('TensorFlow Version:', tf.__version__)
print('GPUs Available:', tf.config.list_physical_devices('GPU'))


In [ ]:
# Step 3: Load Clean Dataset Splits
assert os.path.exists('clean_train.csv'), '❌ Run dataset_analysis.ipynb first to generate clean CSV splits!'

train_df = pd.read_csv('clean_train.csv')
val_df = pd.read_csv('clean_val.csv')
test_df = pd.read_csv('clean_test.csv')

class_names_clean = sorted(train_df['class_name'].unique())
class_to_idx = {name: i for i, name in enumerate(class_names_clean)}
num_classes = len(class_names_clean)

print(f'Clean Classes ({num_classes}):', class_names_clean)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')


In [ ]:
# Step 4: High-Performance tf.data.Dataset Pipeline
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

def load_and_preprocess_img(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

def create_tf_dataset(df, is_training=False):
    paths = df['filepath'].values
    labels = np.array([class_to_idx[c] for c in df['class_name']])
    
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_and_preprocess_img, num_parallel_calls=tf.data.AUTOTUNE)
    
    if is_training:
        # Simple data augmentation
        aug = tf.keras.Sequential([
            layers.RandomFlip('horizontal'),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.1)
        ])
        ds = ds.map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.shuffle(buffer_size=len(df), seed=SEED)
        
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

clean_train_ds = create_tf_dataset(train_df, is_training=True)
clean_val_ds = create_tf_dataset(val_df, is_training=False)
clean_test_ds = create_tf_dataset(test_df, is_training=False)

# Compute balanced class weights
train_counts = Counter(train_df['class_name'])
total_train = len(train_df)
class_weights = {
    class_to_idx[c]: (total_train / (num_classes * train_counts[c]))
    for c in class_names_clean
}
print('✅ tf.data datasets and class weights configured!')


## 🏗️ Model 1: Baseline Custom CNN on Clean Data
Architecture: 3 Conv blocks with `BatchNormalization`, `MaxPooling2D`, and `GlobalAveragePooling2D`.

In [ ]:
# Step 5: Build Baseline Custom CNN Model
baseline_model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
baseline_model.summary()


In [ ]:
# Step 6: Train Baseline Model (10 Epochs)
print('🚀 Starting Baseline Training on Clean Dataset...')
history_baseline = baseline_model.fit(
    clean_train_ds,
    epochs=10,
    validation_data=clean_val_ds,
    class_weight=class_weights
)

# Save baseline model and history
baseline_model.save('clean_baseline_model.h5')
with open('clean_baseline_history.json', 'w') as f:
    json.dump({k: [float(x) for x in v] for k, v in history_baseline.history.items()}, f)
print('✅ Baseline model saved as clean_baseline_model.h5')


## 🚀 Model 2: Fine-Tuning MobileNetV2 on Clean Data
Transfer learning with a two-phase training protocol.

In [ ]:
# Step 7: Build Pretrained MobileNetV2 Architecture
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Phase 1: Freeze base model
base_model.trainable = False

finetuned_model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    # Scale [0, 1] to [-1, 1] for MobileNetV2
    layers.Lambda(lambda x: (x * 2.0) - 1.0),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

finetuned_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
finetuned_model.summary()


In [ ]:
# Step 8: Phase 1 Training - Feature Extraction (5 Epochs)
print('🚀 Starting Phase 1 Training (Frozen Base)...')
history_p1 = finetuned_model.fit(
    clean_train_ds,
    epochs=5,
    validation_data=clean_val_ds,
    class_weight=class_weights
)


In [ ]:
# Step 9: Phase 2 Training - Deep Fine-Tuning Top 30 Layers (7 Epochs)
print('🚀 Starting Phase 2 Deep Fine-Tuning...')
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with small learning rate to adapt features
finetuned_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_p2 = finetuned_model.fit(
    clean_train_ds,
    epochs=12,
    initial_epoch=5,
    validation_data=clean_val_ds,
    class_weight=class_weights
)

# Save fine-tuned model and combined history
finetuned_model.save('clean_finetuned_model.h5')
combined_history = {}
for k in history_p1.history.keys():
    combined_history[k] = [float(x) for x in history_p1.history[k] + history_p2.history[k]]
with open('clean_finetuned_history.json', 'w') as f:
    json.dump(combined_history, f)

# Save class mapping
with open('clean_classes.json', 'w') as f:
    json.dump(class_to_idx, f, indent=4)

print('✅ Fine-tuned model saved as clean_finetuned_model.h5')
print('✅ Class mappings saved as clean_classes.json')


### ✅ Model Development Summary:
1. **Baseline CNN** trained on clean data and saved as `clean_baseline_model.h5`.
2. **Fine-Tuned MobileNetV2** trained in 2 phases on clean data and saved as `clean_finetuned_model.h5`.
3. Models are trained on uncontaminated, deduplicated wheat disease images with zero train-test leakage.

👉 Next Step: Open `model_performance.ipynb` to evaluate both models on `clean_test_ds`!